# PROC: feedback V4 check (business comparison with Every Angle, 16 Nov 2026)

**17 Nov 2026.** The business compared `Fact_PO_LineItem_Schedule` (294,045 rows, POs created since 1 Sep 2026) with Every Angle and
listed duplicates, NULL schedule lines, different execution statuses, missing POs and empty GR quantities
(`Procurment/Fabric_output_V4_20261116.xlsx`; our reading of every difference, groups D1 to D12, is in
`Procurment/Fabric_output_V4_findings.xlsx`). This notebook holds the queries that settle the open groups. It only reads data and
changes nothing; the results stay with the notebook. Run it in the Procurement workspace after Bronze, Silver and `NB_Gold_PROC`.

| Query | Settles | Reads |
|---|---|---|
| Q1 | D1: how many lines have no goods receipt in Fabric although Every Angle shows one | Silver EKET, EKPO, EKKO, EKBE |
| Q2 | D1 / D10: do Bronze EKET and EKBE still receive changes | Bronze, OPS `table_log` |
| Q3 | D1: the example POs through every layer; at which layer the receipt disappears | Bronze → Silver → Gold, DAB source if `SOURCE_ABFSS` is set |
| Q4 | D1: a key problem between EKBE and EKPO | Silver |
| Q5 | D3: zero-quantity schedule lines, and the ones that show a receipt | Silver EKET, EKBE |
| Q6 | D5: Closed by GR where Every Angle says Outbound delivery created (stock transfers) | Gold, Silver, SD Silver LIPS |
| Q7 | D6: the four open-line status pairs | Gold, Silver EKES / EKET / EKPO |
| Q8 | D10: the PO Every Angle has and Fabric lacks | Bronze EKKO, EKPO, EKET |
| Q9 | D7 / D8: duplicated keys and over-confirmed pieces (0 in the fact after 17 Nov) | Gold |
| Q10 | D2: how many lines would change with a line-level execution status | Gold |
| Q12 | Line-level comparison with the 16 Nov extract (19 Nov 2026): the extract's Every Angle values (uploaded file) against today's order and execution status and the execution status with Option A; prints totals, per-cause tables and example lines; a table only when `SAVE_COMPARE_TABLE` is True | Uploaded `EA_FILE`, Gold, Silver EKET / EKPO / EKBE |
| Q13 | The evidence behind each remaining difference (19 Nov 2026): SAP changes from 16 Nov, execution-status pairs with the D2 line status, stock-transfer receipts, delivery completed / tolerance / invoice for causes 6 and 8, EKES ZB / ZN for cause 3, deletion flags and reversed receipts, zero-quantity lines | Q12 result, Silver EKPO / EKKO / EKBE / EKES |
| Q14 | The 18 Nov audit fields after the Gold run: fill rates, header totals against items, Invoice_Status, First_GR_Date, DQ results of the last PROC Gold run | Gold, OPS `dq_results` |
| Q15 | Causes 6 and 8: candidate rules (FIFO receipts, item fully received, MRP reduction, invoices; no goods receipt expected, fully invoiced, service, account assignment) scored against a control group | Q12 result, Silver EKET / EKPO / EKBE |

Decisions D7 and D8 were taken on 17 Nov 2026 (one row per schedule line; over-confirmed pieces only in the evidence table).
D2 to D6 wait for Jeremy and Procurement team; D1 and D10 are data questions for the A/I team.


In [1]:
# ── Parameters ─────────────────────────────────────────────────────────────────────────────────────────────
PIPELINE_RUN_ID = ''
EXAMPLE_POS     = ['13219558', '13219560', '13219566', '13219607', '13219824', '13219834', '13148876', '4508252311']
CREATED_FROM    = '2026-09-01'      # the business extract: POs created on or after this date (EKKO AEDAT)
EXTRACT_TO      = '2026-11-14'      # ▶ 19 Nov 2026: last creation date in the 16 Nov extract (Q12 exports the same POs)
EA_FILE         = 'Files/PROC_EA_16Nov_upload.csv'   # ▶ 19 Nov 2026: Q12 input, uploaded to the PROC Gold lakehouse Files
N_LINE_EXAMPLES = 10                # ▶ 19 Nov 2026: Q12 example lines printed per result and cause
SAVE_COMPARE_TABLE = False          # ▶ 19 Nov 2026: True writes Q12's line list to Gold table Chk_PROC_Order_Status_Compare (for SQL)
SOURCE_ABFSS    = ''                # optional: abfss path of the DAB lakehouse that feeds PROC Bronze, to compare EKET / EKBE at the source
SOURCE_SCHEMA   = 'dbo'
N_EXAMPLES      = 20

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 3, Finished, Available, Finished, False)

In [2]:
%run ./NB_Utils_Common

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 9, Finished, Available, Finished, True)

NB_Utils_Common: settings ready
NB_Utils_Common: locations ready
NB_Utils_Common: start_run() ready
NB_Utils_Common: OPS logging ready
NB_Utils_Common: dq_record(), dq_flush(), check_volume_change() ready
NB_Utils_Common loaded: start_run(), tpath(), record_table(), dq_record(), end_of_run_report(), end_session()


In [3]:
%run ./NB_Config_PROC

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 16, Finished, Available, Finished, True)

Config loaded: ['EKKO', 'EKPO', 'EKET', 'EKBE']
EA audit report tables added: EKES, EKKN, EKPA, EKAB
New tables added:EKPV, EBKN, EBAN


In [4]:
%run ./NB_Utils_Gold

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 17, Finished, Available, Finished, True)

NB_Utils_Gold loaded: silver(), gold(), drop_redundant(), sk_lookup(), write_gold(), stable_sk(), promote_to_gold(), profile_strings()


In [5]:
# ── Setup ──────────────────────────────────────────────────────────────────────────────────────────────────
start_run('PROC_Check_Feedback_V4', 'CHECK')
fact  = gold('Fact_PO_LineItem_Schedule')
ev    = gold('Fact_PO_Order_Status_Evidence')
eket, ekpo, ekko, ekbe, ekes = silver('EKET'), silver('EKPO'), silver('EKKO'), silver('EKBE'), silver('EKES')
_nz   = lambda c: F.regexp_replace(F.col(c).cast('string'), '^0+', '')
# the POs of the extract: created on or after CREATED_FROM
pos = ekko.filter(F.col('Created_On') >= F.lit(CREATED_FROM)).select('Purchasing_Document', 'Created_On', 'Purch_Doc_Category')
# goods receipts per PO item (physical GR: VGABE 1 / BEWTP E), as cell 20 of NB_Gold_PROC counts them
gr_item = (ekbe.filter(F.col('Transevent_Type').eqNullSafe('1') | F.col('PO_History_Category').eqNullSafe('E'))
           .groupBy('Purchasing_Document', 'Item')
           .agg(F.sum(F.col('Signed_Quantity').cast('double')).alias('Item_Gr_Qty'), F.max('Posting_Date').alias('Last_GR_Date'),
                F.count('*').alias('GR_Rows')))
for _n, _df in [('fact', fact), ('ev', ev), ('eket', eket), ('ekpo', ekpo), ('ekko', ekko), ('ekbe', ekbe), ('ekes', ekes),
                ('pos', pos), ('gr_item', gr_item)]:
    _df.createOrReplaceTempView(_n)
print(f"Fact_PO_LineItem_Schedule: {fact.count():,} rows | POs created since {CREATED_FROM}: {pos.count():,}")
_ex = ', '.join(f"'{p}'" for p in EXAMPLE_POS)

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 18, Finished, Available, Finished, False)

Pipeline : PROC_Check_Feedback_V4
Run ID   : 618b50e1-caec-4134-850e-198c55001fa4
Started  : 2026-11-20 00:13:36 UTC
Locations:
  bronze     dbo              abfss://3532debe-0172-4493-815c-dc7aed901ac7@onelake.dfs.fabric.micros…
  silver     dbo              abfss://3532debe-0172-4493-815c-dc7aed901ac7@onelake.dfs.fabric.micros…
  gold       dbo              abfss://3532debe-0172-4493-815c-dc7aed901ac7@onelake.dfs.fabric.micros…
  md_silver  dbo              abfss://9cb5a2e3-2121-4f0d-8289-c1f350ac646f@onelake.dfs.fabric.micros…
  md_gold    dbo              abfss://9cb5a2e3-2121-4f0d-8289-c1f350ac646f@onelake.dfs.fabric.micros…
  ops        dbo              abfss://548c95f1-2c7c-44e4-a26f-e6971f80339f@onelake.dfs.fabric.micros…
  sd_silver  dbo              abfss://303715d4-7108-4456-a1f2-bbd04c28cc0c@onelake.dfs.fabric.micros…
  xrate      Exchange_Rates   abfss://9cb5a2e3-2121-4f0d-8289-c1f350ac646f@onelake.dfs.fabric.micros…
Fact_PO_LineItem_Schedule: 6,124,697 rows | POs created 

### Q1 · D1 — lines with no goods receipt in Fabric
**Issue:** 85,777 lines are Open in Fabric and Closed by GR in Every Angle; all have EKET WEMNG 0 and no EKBE row. **Expected:** about the same number here if Fabric lacks the receipts. **Feeds:** whether the A/I team must fix the EKET / EKBE extract.

In [6]:
# ── Q1 · D1: schedule lines with no goods receipt in Fabric, by month and plant ────────────────────────────────
# Every Angle closes them by GR. Expected: about 86,000 lines (the extract) when Fabric lacks the receipts; far fewer when the
# extract was taken from an old run and Fabric has caught up since.
q1 = spark.sql(f'''
SELECT date_format(p.Created_On, 'yyyy-MM') AS created_month, i.Plant,
       SUM(CASE WHEN COALESCE(l.Quantity_Delivered, 0) = 0 AND g.Item_Gr_Qty IS NULL AND COALESCE(l.Scheduled_Quantity, 0) > 0 THEN 1 ELSE 0 END) AS lines_no_receipt,
       SUM(CASE WHEN COALESCE(l.Quantity_Delivered, 0) > 0 OR g.Item_Gr_Qty IS NOT NULL THEN 1 ELSE 0 END)   AS lines_with_receipt,
       COUNT(*) AS lines
FROM eket l JOIN pos p ON p.Purchasing_Document = l.Purchasing_Document
LEFT JOIN ekpo i ON i.Purchasing_Document = l.Purchasing_Document AND i.Item = l.Item
LEFT JOIN gr_item g ON g.Purchasing_Document = l.Purchasing_Document AND g.Item = l.Item
GROUP BY 1, 2 ORDER BY 1, 2''')
display(q1)
_tot = q1.agg(F.sum('lines_no_receipt'), F.sum('lines_with_receipt')).first()
print(f'Lines without any receipt in Fabric: {_tot[0] or 0:,} | with a receipt: {_tot[1] or 0:,}')

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 348646fe-95f1-4f49-a03d-844887aa0644)

Lines without any receipt in Fabric: 65,502 | with a receipt: 236,991


### Q2 · D1 / D10 — Bronze freshness
**Issue:** goods receipts update existing EKET rows and add EKBE rows. **Expected:** recent `_UPDATED_AT`, `rows_changed > 0` for EKET. **Feeds:** extract diagnosis.

In [7]:
# ── Q2 · D1 / D10: are Bronze EKKO, EKET and EKBE still being refreshed? ──────────────────────────────────────
# Expected: _UPDATED_AT close to the last run for all three; rows_changed > 0 for EKET on recent runs (goods receipts update
# WEMNG on existing EKET rows). A table whose rows_changed is always 0 only receives inserts.
for _t in ['EKKO', 'EKET', 'EKBE']:
    _b = read_table('bronze', _t)
    _r = _b.agg(F.count('*'), F.max('_UPDATED_AT'), F.sum(F.col('_IS_DELETED').cast('int'))).first()
    print(f'Bronze {_t}: {_r[0]:,} rows, last _UPDATED_AT {_r[1]}, deleted {_r[2] or 0:,}')
    try:
        display(spark.sql(f"DESCRIBE HISTORY delta.`{tpath('bronze', _t)}`").select('version', 'timestamp', 'operation', 'operationMetrics').limit(5))
    except Exception as e:
        print(f'  history not readable ({str(e)[:100]})')
try:
    _log = read_table('ops', 'table_log')
    display(_log.filter(F.col('table_name').isin('EKKO', 'EKET', 'EKBE') & (F.col('logged_at') >= F.lit('2026-11-01')))
                .select('logged_at', 'layer', 'table_name', 'status', 'load_mode', 'rows_source', 'rows_in', 'rows_new', 'rows_changed',
                        'rows_deleted', 'rows_out', 'duration_s')
                .orderBy(F.desc('logged_at')))
except Exception as e:
    print(f'OPS table_log not readable ({str(e)[:120]})')

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 20, Finished, Available, Finished, False)

Bronze EKKO: 3,401,432 rows, last _UPDATED_AT 2026-11-19 08:18:15.750734, deleted 0


SynapseWidget(Synapse.DataFrame, cf0372a6-44a7-48d5-96aa-4d42ce35c8bc)

Bronze EKET: 6,914,732 rows, last _UPDATED_AT 2026-11-19 08:15:36.154512, deleted 0


SynapseWidget(Synapse.DataFrame, 9f517f2a-8190-4a49-a2f4-2850a8b88981)

Bronze EKBE: 17,889,092 rows, last _UPDATED_AT 2026-11-19 08:18:29.050113, deleted 0


SynapseWidget(Synapse.DataFrame, 35b351df-8c06-4a35-8e5a-bb6061f13938)

SynapseWidget(Synapse.DataFrame, b2ad0439-5757-4159-90b8-6f775c41f28f)

### Q3 · D1 — example POs through every layer
**Issue:** where does the receipt disappear. **Expected:** Bronze already lacks it (source gap). **Feeds:** who fixes it.

In [8]:
# ── Q3 · D1: the example POs through every layer ─────────────────────────────────────────────────────────────
# Expected (if D1 is a source gap): EKET WEMNG 0 and no EKBE rows in Bronze already. If Bronze has the receipt and Silver or
# Gold does not, the loss is ours (then see Q4).
_br_eket = read_table('bronze', 'EKET').filter(_nz('EBELN').isin([p.lstrip('0') for p in EXAMPLE_POS]))
display(_br_eket.select('EBELN', 'EBELP', 'ETENR', 'EINDT', 'MENGE', 'WEMNG', 'DABMG', 'MNG02', '_UPDATED_AT', '_IS_DELETED').orderBy('EBELN', 'EBELP', 'ETENR'))
_br_ekbe = read_table('bronze', 'EKBE').filter(_nz('EBELN').isin([p.lstrip('0') for p in EXAMPLE_POS]))
print(f'Bronze EKBE rows for the examples: {_br_ekbe.count():,}')
display(_br_ekbe.select('EBELN', 'EBELP', 'VGABE', 'BEWTP', 'BWART', 'BUDAT', 'MENGE', 'SHKZG', 'ETENS', '_UPDATED_AT').orderBy('EBELN', 'EBELP', 'BUDAT'))
display(spark.sql(f'''
SELECT l.Purchasing_Document, l.Item, l.Schedule_Line, l.Scheduled_Quantity, l.Quantity_Delivered, l.Qty_Reduced_MRP, l.Committed_Quantity,
       i.Delivery_Completed, g.Item_Gr_Qty, g.GR_Rows, g.Last_GR_Date
FROM eket l LEFT JOIN ekpo i ON i.Purchasing_Document = l.Purchasing_Document AND i.Item = l.Item
LEFT JOIN gr_item g ON g.Purchasing_Document = l.Purchasing_Document AND g.Item = l.Item
WHERE l.Purchasing_Document IN ({_ex}) ORDER BY 1, 2, 3'''))
display(fact.filter(F.col('Purchasing_Document').isin(EXAMPLE_POS))
            .select('Purchasing_Document', 'Item', 'Schedule_Line', 'Order_Status_Purchase', 'Execution_Status', 'Scheduled_Quantity',
                    'Quantity_Delivered', 'Realized_Finish_Date', 'Piece_Count', 'Confirmation_Categories')
            .orderBy('Purchasing_Document', 'Item', 'Schedule_Line'))
if SOURCE_ABFSS:
    for _t, _cols in [('EKET', ['EBELN', 'EBELP', 'ETENR', 'MENGE', 'WEMNG', 'DABMG']), ('EKBE', ['EBELN', 'EBELP', 'VGABE', 'BWART', 'BUDAT', 'MENGE'])]:
        try:
            _s = spark.read.format('delta').load(f'{SOURCE_ABFSS}/Tables/{SOURCE_SCHEMA}/{_t}')
            display(_s.filter(_nz('EBELN').isin([p.lstrip('0') for p in EXAMPLE_POS])).select(*_cols))
        except Exception as e:
            print(f'Source {_t} not readable ({str(e)[:120]})')
else:
    print('SOURCE_ABFSS not set: the DAB source is not compared (the A/I team can run the same selection there)')

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 623fd9ec-cd5b-4973-9b4d-ebf3e63c4e24)

Bronze EKBE rows for the examples: 29


SynapseWidget(Synapse.DataFrame, bd9ca472-5673-4aa1-8a8a-29b65fe8d32f)

SynapseWidget(Synapse.DataFrame, 63e571cd-d21d-43e8-a48f-974ecca43df7)

SynapseWidget(Synapse.DataFrame, 3f4d5033-7ee7-4914-84d6-426491169602)

SOURCE_ABFSS not set: the DAB source is not compared (the A/I team can run the same selection there)


### Q4 · D1 — EKBE keys
**Issue:** a leading-zero or type mismatch would drop receipts in Silver / Gold. **Expected:** 0 unmatched, one key length. **Feeds:** a config fix only if this fails.

In [9]:
# ── Q4 · D1: do the EKBE keys match EKPO? ────────────────────────────────────────────────────────────────────
# Expected: 0 unmatched rows and one key length per table. Unmatched rows or two lengths mean a leading-zero / type problem.
display(spark.sql('''
SELECT COUNT(*) AS ekbe_rows,
       SUM(CASE WHEN i.Purchasing_Document IS NULL THEN 1 ELSE 0 END) AS no_ekpo_match
FROM ekbe b LEFT JOIN ekpo i ON i.Purchasing_Document = b.Purchasing_Document AND i.Item = b.Item'''))
for _n in ['ekbe', 'eket', 'ekpo']:
    display(spark.sql(f'''SELECT '{_n}' AS tbl, LENGTH(Purchasing_Document) AS po_len, LENGTH(Item) AS item_len, COUNT(*) AS rows
                          FROM {_n} GROUP BY 1, 2, 3 ORDER BY 4 DESC'''))

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c64b60f3-d8dd-443e-981b-6ca09d635ccd)

SynapseWidget(Synapse.DataFrame, 45222095-765a-4cb3-b4b6-26e5e3963ced)

SynapseWidget(Synapse.DataFrame, e00364d0-25b9-4346-a9ba-99ad7be30cf9)

SynapseWidget(Synapse.DataFrame, a727d3b2-48c3-4182-ba18-608450d60e78)

### Q5 · D3 — zero-quantity schedule lines
**Issue:** 27,392 items end with a line of quantity 0 that Every Angle hides; 7,940 of those lines show a receipt. **Expected:** the committed quantity equals the item's scheduled quantity; receipts on zero lines explained. **Feeds:** business decision D3.

In [10]:
# ── Q5 · D3: zero-quantity schedule lines ────────────────────────────────────────────────────────────────────
# Expected: on most items the zero line is the last one and its committed quantity equals the other lines' scheduled quantity
# (an EKET row carrying the ATP commitment; Every Angle hides it). A zero line with WEMNG > 0 should not exist: list them.
_zl = spark.sql('''
WITH l AS (SELECT Purchasing_Document, Item, Schedule_Line, COALESCE(Scheduled_Quantity, 0) AS sched, COALESCE(Quantity_Delivered, 0) AS recv,
                  COALESCE(Committed_Quantity, 0) AS com, MAX(Schedule_Line) OVER (PARTITION BY Purchasing_Document, Item) AS last_line,
                  SUM(COALESCE(Scheduled_Quantity, 0)) OVER (PARTITION BY Purchasing_Document, Item) AS item_sched
           FROM eket)
SELECT COUNT(*) AS zero_lines,
       SUM(CASE WHEN Schedule_Line = last_line THEN 1 ELSE 0 END)            AS zero_is_last_line,
       SUM(CASE WHEN ABS(com - item_sched) < 0.001 THEN 1 ELSE 0 END)        AS committed_equals_item_scheduled,
       SUM(CASE WHEN recv > 0 THEN 1 ELSE 0 END)                             AS zero_lines_with_receipt
FROM l WHERE sched = 0 AND com > 0''')
display(_zl)
display(spark.sql(f'''
SELECT l.Purchasing_Document, l.Item, l.Schedule_Line, l.Scheduled_Quantity, l.Quantity_Delivered, l.Committed_Quantity, l.Committed_Date,
       l.Delivery_Date, g.Item_Gr_Qty, g.GR_Rows
FROM eket l LEFT JOIN gr_item g ON g.Purchasing_Document = l.Purchasing_Document AND g.Item = l.Item
WHERE COALESCE(l.Scheduled_Quantity, 0) = 0 AND COALESCE(l.Quantity_Delivered, 0) > 0
ORDER BY l.Purchasing_Document, l.Item LIMIT {N_EXAMPLES}'''))

# ▶ 18 Nov 2026 (first run): 720,937 zero lines show a receipt. In 13219607 item 2 both lines carry WEMNG 1,200 and EKBE has 1,200,
#   so adding Quantity_Delivered over an item's lines counts the receipt twice. How many items does that affect?
display(spark.sql('''
WITH s AS (SELECT Purchasing_Document, Item, SUM(COALESCE(Quantity_Delivered, 0)) AS line_recv,
                  MAX(CASE WHEN COALESCE(Scheduled_Quantity, 0) = 0 AND COALESCE(Quantity_Delivered, 0) > 0 THEN 1 ELSE 0 END) AS zero_line_recv
           FROM eket GROUP BY Purchasing_Document, Item)
SELECT s.zero_line_recv = 1 AS has_zero_line_with_receipt, COUNT(*) AS items,
       SUM(CASE WHEN g.Item_Gr_Qty IS NOT NULL AND s.line_recv > g.Item_Gr_Qty + 0.001 THEN 1 ELSE 0 END) AS items_lines_above_ekbe_gr,
       SUM(CASE WHEN g.Item_Gr_Qty IS NOT NULL AND ABS(s.line_recv - 2 * g.Item_Gr_Qty) <= 0.001 AND g.Item_Gr_Qty > 0 THEN 1 ELSE 0 END) AS items_lines_twice_ekbe_gr
FROM s LEFT JOIN gr_item g ON g.Purchasing_Document = s.Purchasing_Document AND g.Item = s.Item
GROUP BY 1 ORDER BY 1'''))


StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9341dc0b-70b0-4e43-8157-76591855567a)

SynapseWidget(Synapse.DataFrame, 80ab8559-131e-44ac-9735-f0575042bbd0)

SynapseWidget(Synapse.DataFrame, d41e6cd2-684e-4816-89ac-be83ca5575df)

### Q6 · D5 — Closed by GR vs Outbound delivery created
**Issue:** 8,439 fully received stock-transfer lines. **Expected:** PSTYP U with a delivery. **Feeds:** rule order for stock transfers (business).

In [11]:
# ── Q6 · D5: Closed by GR lines on stock transfers (Every Angle: Outbound delivery created) ───────────────────
# Expected: PSTYP U items, fully received, with an outbound delivery (LIPS) or an inbound delivery on the confirmation. The
# question for the business is the rule order for these lines (rule 2 Closed by GR before rule 10 today).
_sto = (fact.filter(F.col('Order_Status_Purchase') == 'Closed by GR')
            .join(ekpo.select('Purchasing_Document', 'Item', 'Item_Category_Ext'), ['Purchasing_Document', 'Item'], 'left')   # Delivery_Completed is on the fact already (18 Nov 2026)
            .filter(F.trim(F.col('Item_Category_Ext')).eqNullSafe('U')))
print(f"Closed by GR on stock-transfer items (PSTYP U): {_sto.count():,} lines")
_mv = ekbe.groupBy('Purchasing_Document', 'Item').agg(F.concat_ws('|', F.sort_array(F.collect_set('Movement_Type'))).alias('Movement_Types'))
_ib = ekes.groupBy('Purchasing_Document', 'Item').agg(F.max(F.when(F.trim(F.coalesce(F.col('Inbound_Delivery'), F.lit(''))) != '', F.col('Inbound_Delivery'))).alias('Inbound_Delivery'))
_q6 = (_sto.join(_mv, ['Purchasing_Document', 'Item'], 'left').join(_ib, ['Purchasing_Document', 'Item'], 'left'))
if 'sd_silver' in LOC and table_exists('sd_silver', 'LIPS'):
    _lips = (read_table('sd_silver', 'LIPS', active=True).select(_nz('Reference_Document').alias('Purchasing_Document'), _nz('Reference_Item').alias('Item'))
             .dropDuplicates().withColumn('Has_LIPS_Delivery', F.lit(True)))
    _q6 = _q6.join(_lips, ['Purchasing_Document', 'Item'], 'left')
else:
    print('SD Silver LIPS not reachable: Has_LIPS_Delivery not shown')
    _q6 = _q6.withColumn('Has_LIPS_Delivery', F.lit(None).cast('boolean'))
display(_q6.join(pos.select('Purchasing_Document'), 'Purchasing_Document')   # ▶ 18 Nov 2026: examples from the extract's POs
           .select('Purchasing_Document', 'Item', 'Schedule_Line', 'Execution_Status', 'Scheduled_Quantity', 'Quantity_Delivered',
                   'Qty_Delivered_STO', 'Committed_Quantity', 'Delivery_Completed', 'Movement_Types', 'Inbound_Delivery', 'Has_LIPS_Delivery')
           .orderBy('Purchasing_Document', 'Item', 'Schedule_Line').limit(N_EXAMPLES))

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 24, Finished, Available, Finished, False)

Closed by GR on stock-transfer items (PSTYP U): 1,899,611 lines


SynapseWidget(Synapse.DataFrame, 4fb8c094-043f-41ea-bbd6-92b2d5455e7d)

### Q7 · D6 — open-line status pairs
**Issue:** four pairs where both sides see an open line but name it differently. **Expected:** the EKES rows and committed quantities show which evidence Every Angle used. **Feeds:** business decisions on rules 10, 11, 14, 15.

In [12]:
# ── Q7 · D6: the four open-line status pairs — what evidence each side sees ─────────────────────────────────
# Fabric → Every Angle: To be confirmed → Stock committed (2,003 in the extract); Stock committed → Outbound delivery created
# (1,081); Shipping notification → Confirmed (945); To be confirmed → Outbound delivery created (1,166). For N_EXAMPLES lines of
# each Fabric status: the item's EKES rows (category, dates, quantities), committed quantity and date, PSTYP, BSTAE / KZABS / LABNR.
_conf = (ekes.filter(F.trim(F.coalesce(F.col('Confirmation_Deletion'), F.lit(''))) == '')
             .groupBy('Purchasing_Document', 'Item')
             .agg(F.count('*').alias('EKES_Rows'),
                  F.concat_ws('|', F.sort_array(F.collect_set('Confirmation_Category'))).alias('EKES_Categories'),
                  F.max('Confirmed_Delivery_Date').alias('Last_Confirmed_Date'), F.max('Created_On').alias('Last_Conf_Created'),
                  F.sum(F.col('Confirmation_Quantity').cast('double')).alias('Conf_Qty'),
                  F.sum(F.col('Confirmation_Qty_Reduced_MRP').cast('double')).alias('Conf_Qty_Reduced'),
                  F.max(F.when(F.trim(F.coalesce(F.col('Inbound_Delivery'), F.lit(''))) != '', F.col('Inbound_Delivery'))).alias('Inbound_Delivery')))
_itm = ekpo.select('Purchasing_Document', 'Item', 'Item_Category_Ext', 'Confirmation_Control', 'Order_Acknowledgment')   # Acknowledgment_Reqd is on the fact already (18 Nov 2026)
for _status in ['To be confirmed', 'Stock committed', 'Shipping notification']:
    print(f'\n{_status}: open lines, {N_EXAMPLES} examples')
    display(fact.filter((F.col('Order_Status_Purchase') == _status) & (F.coalesce(F.col('Quantity_Delivered'), F.lit(0)) == 0))
                .join(pos.select('Purchasing_Document'), 'Purchasing_Document')   # ▶ 18 Nov 2026: the extract's POs (the first run showed 2018-2022 lines)
                .join(_conf, ['Purchasing_Document', 'Item'], 'left').join(_itm, ['Purchasing_Document', 'Item'], 'left')
                .select('Purchasing_Document', 'Item', 'Schedule_Line', 'Delivery_Date', 'Scheduled_Quantity', 'Committed_Quantity', 'Committed_Date',
                        'Item_Category_Ext', 'Confirmation_Control', 'Acknowledgment_Reqd', 'Order_Acknowledgment',
                        'EKES_Rows', 'EKES_Categories', 'Last_Confirmed_Date', 'Last_Conf_Created', 'Conf_Qty', 'Conf_Qty_Reduced', 'Inbound_Delivery',
                        'Piece_Count', 'Confirmation_Categories')
                .orderBy('Purchasing_Document', 'Item', 'Schedule_Line').limit(N_EXAMPLES))

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 25, Finished, Available, Finished, False)


To be confirmed: open lines, 20 examples


SynapseWidget(Synapse.DataFrame, edf1df96-f934-4dbc-ba04-7f3363c62270)


Stock committed: open lines, 20 examples


SynapseWidget(Synapse.DataFrame, 75a51348-c9b3-46e4-9b07-66038b908256)


Shipping notification: open lines, 20 examples


SynapseWidget(Synapse.DataFrame, fb468364-4052-45cb-9d46-e19d60bd9415)

### Q8 · D10 — missing PO
**Issue:** PO 13148876 (created 24 Sep) is in Every Angle, not in Fabric. **Expected:** absent from Bronze or flagged deleted. **Feeds:** the A/I team.

In [13]:
# ── Q8 · D10: the PO Every Angle has and Fabric lacks ───────────────────────────────────────────────────────
# Expected: no row in Bronze EKKO (extract gap) or a row with _IS_DELETED = true (deleted upstream). The business should send
# the full list of missing POs; EXAMPLE_POS has the one example from the file (13148876).
for _t, _key in [('EKKO', 'EBELN'), ('EKPO', 'EBELN'), ('EKET', 'EBELN')]:
    _b = read_table('bronze', _t).filter(_nz(_key).isin([p.lstrip('0') for p in EXAMPLE_POS]))
    display(_b.groupBy(_key, '_IS_DELETED').agg(F.count('*').alias('rows'), F.min('_CREATED_AT').alias('first_loaded'), F.max('_UPDATED_AT').alias('last_updated'))
              .withColumn('table', F.lit(_t)).orderBy(_key))

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c66720a-220a-4ff5-9c3c-f259106798a7)

SynapseWidget(Synapse.DataFrame, 5325dbe2-4af3-42fe-bc07-d5fa21a9f47a)

SynapseWidget(Synapse.DataFrame, 1066ed17-a36c-4015-89e3-de312c4a7cd8)

### Q9 · D7 / D8 — duplicates and over-confirmed pieces
**Expected after the 17 Nov change:** 0 keys with more than one row, 0 rows without a schedule line; the pieces are in the evidence table.

In [14]:
# ── Q9 · D7 / D8: duplicated keys and over-confirmed pieces ─────────────────────────────────────────────────
# Before 17 Nov 2026 the fact was one row per segment: a partly confirmed line had two rows (259 keys in the extract) and the
# over-confirmed pieces had no schedule line (4,398 rows). Expected now: 0 and 0 in the fact; the pieces are in the evidence table.
_dup = fact.groupBy('Purchasing_Document', 'Item', 'Schedule_Line').count().filter('count > 1')
print(f"Keys with more than one row in the fact: {_dup.count():,} (expected 0) | rows with an empty Schedule_Line: "
      f"{fact.filter(F.col('Schedule_Line').isNull()).count():,} (expected 0)")
print(f"Evidence table: {ev.count():,} segments; over-confirmed pieces (no schedule line): "
      f"{ev.filter(F.coalesce(F.col('Is_Over_Confirmed'), F.lit(False))).count():,}; lines with more than one piece: "
      f"{fact.filter(F.col('Piece_Count') > 1).count():,}")
display(fact.filter(F.col('Piece_Count') > 1)
            .select('Purchasing_Document', 'Item', 'Schedule_Line', 'Order_Status_Purchase', 'Piece_Count', 'Confirmation_Categories', 'Segment_Quantity')
            .orderBy('Purchasing_Document', 'Item', 'Schedule_Line').limit(N_EXAMPLES))

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 27, Finished, Available, Finished, False)

Keys with more than one row in the fact: 0 (expected 0) | rows with an empty Schedule_Line: 0 (expected 0)
Evidence table: 6,934,450 segments; over-confirmed pieces (no schedule line): 18,414; lines with more than one piece: 631


SynapseWidget(Synapse.DataFrame, c3fd1db1-21e6-44c0-a096-8b5efcbb7baa)

### Q10 · D2 — line-level execution status
**Issue:** 9,292 lines fully received show Partially Open because the status is the item's. **Expected:** those lines become Closed. **Feeds:** business decision D2.

In [15]:
# ── Q10 · D2: a line-level execution status against the item-level one ───────────────────────────────────────
# Proposal D2 (NB_Gold_PROC cell 22, commented): Cancelled when the item is; Closed when ELIKZ is set or nothing is left to receive;
# Partially Open when some is received; else Open. Expected: the Partially Open → Closed pairs (9,292 in the extract) move.
_recv, _sched = F.coalesce(F.col('Quantity_Delivered'), F.lit(0)), F.coalesce(F.col('Scheduled_Quantity'), F.lit(0))
_line_status = (F.when(F.col('Execution_Status').isin('Cancelled_Header', 'Cancelled_Line'), F.col('Execution_Status'))
                 .when((F.col('Delivery_Completed') == 'X') | ((_sched - _recv) <= OS_TOLERANCE), 'Closed')
                 .when(_recv > 0, 'Partially Open').otherwise('Open'))
_q10 = fact.withColumn('Line_Execution_Status', _line_status)
display(_q10.groupBy('Execution_Status', 'Line_Execution_Status').count().orderBy(F.desc('count')))
print(f"Lines whose status would change: {_q10.filter(F.col('Execution_Status') != F.col('Line_Execution_Status')).count():,} of {fact.count():,}")

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 65f5ae08-22e0-4c30-80a3-10014a77ed06)

Lines whose status would change: 3,095 of 6,124,697


### Q11 · D1 — time pattern of the missing receipts
**Issue:** are receipts missing since one date (a cut-off) or here and there (sporadic)? **Expected:** a steady weekly count of goods receipts and of changed EKET rows means sporadic; a sudden drop means a cut-off from that week. Compare the weekly counts with the source / Every Angle. **Feeds:** what the A/I team looks for in the DAB extract.

In [16]:
# ── Q11 · D1: goods receipts per posting week, and Bronze rows changed per day (18 Nov 2026) ──────────────────────
# Physical GR (VGABE 1 / BEWTP E) in Silver EKBE by posting week since 1 Aug, and the share of items of the extract's POs that have one.
_gr = ekbe.filter(F.col('Transevent_Type').eqNullSafe('1') | F.col('PO_History_Category').eqNullSafe('E'))
display(_gr.filter(F.col('Posting_Date') >= F.lit('2026-08-01'))
           .groupBy(F.date_trunc('week', F.col('Posting_Date')).cast('date').alias('posting_week'))
           .agg(F.count('*').alias('gr_rows'), F.countDistinct('Purchasing_Document', 'Item').alias('items_with_gr'),
                F.max('Posting_Date').alias('last_posting_date'))
           .orderBy('posting_week'))
# When Bronze last received each row: new or changed EKET / EKBE rows per load day (a gap of days = no load; a low count = little change)
for _t in ('EKET', 'EKBE'):
    print(f'Bronze {_t}: rows by the day they were last loaded or changed (since 1 Sep)')
    display(read_table('bronze', _t).filter(F.col('_UPDATED_AT') >= F.lit('2026-09-01'))
                .groupBy(F.to_date('_UPDATED_AT').alias('load_day')).count().orderBy('load_day'))


StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 787c5266-d34f-4466-b817-03b0b988a844)

Bronze EKET: rows by the day they were last loaded or changed (since 1 Sep)


SynapseWidget(Synapse.DataFrame, d7fd70f6-b5ac-4986-ac9c-6dda84e32b17)

Bronze EKBE: rows by the day they were last loaded or changed (since 1 Sep)


SynapseWidget(Synapse.DataFrame, fa655b87-e5f8-4803-9614-a8fd8600b0a2)

### Q12 · line-level comparison with the 16 Nov extract
`Procurment/PROC_EA_16Nov_upload.csv` holds one row per line of the 16 Nov extract (289,398 lines): Every Angle's order and execution
status, Fabric's 16 Nov values and the 16 Nov cause (1–13, as in `PROC_Order_Status_Differences.xlsx`). Upload it to the PROC Gold
lakehouse under `Files/checks` (Lakehouse explorer → Files → Upload). Q12 joins it to today's `Fact_PO_LineItem_Schedule`, re-derives the
cause, adds the execution status with Option A (`NB_Gold_PROC` cell 13, 19 Nov 2026) and prints everything as text, so the exported
notebook carries the results. With `SAVE_COMPARE_TABLE = True` the full line list is also written to Gold `Chk_PROC_Order_Status_Compare`
for SQL queries.

In [17]:
# ── Q12 · line-level comparison: 16 Nov extract (Every Angle + Fabric then) against Fabric now, with Option A (19 Nov 2026) ──────────
# Option A is re-derived from Silver EKET so the result shows its effect before and after NB_Gold_PROC cell 13 runs with it: Delivered
# adds WEMNG only over lines with a quantity (a zero line keeps its WEMNG only when it holds more than that sum); an item whose lines are
# all zero keeps the sum over all its lines. Last_GR_Date on or after 16 Nov = SAP changed after the extract, so EA may have moved too.
_root = LOC['gold'][0]
_root = _root[:_root.rfind('/Tables')] if '/Tables' in _root else _root
_ea = spark.read.option('header', True).csv(f'{_root}/{EA_FILE}')
_k  = lambda df, po, it, sl: df.withColumn('_po', F.regexp_replace(F.trim(F.col(po)), '^0+', '')) \
                               .withColumn('_it', F.regexp_replace(F.trim(F.col(it)), '^0+', '')) \
                               .withColumn('_sl', F.regexp_replace(F.trim(F.col(sl)), '^0+', ''))
_ea = _k(_ea, 'PO', 'Item', 'Schedule_Line')
_has = F.coalesce(F.col('Scheduled_Quantity'), F.lit(0)) != 0
_item = (eket.groupBy('Purchasing_Document', 'Item')
             .agg(F.sum('Quantity_Delivered').alias('Item_Delivered_All_Lines'),
                  F.sum(F.when(_has, F.col('Quantity_Delivered'))).alias('_q'),
                  F.max(F.when(~_has, F.col('Quantity_Delivered'))).alias('_z'),
                  F.max(_has.cast('int')).alias('_any'),
                  F.sum('Scheduled_Quantity').alias('Item_Scheduled_Qty'))
             .withColumn('Item_Delivered_A', F.when(F.col('_any') == 1, F.greatest('_q', '_z')).otherwise(F.col('Item_Delivered_All_Lines')))
             .drop('_q', '_z', '_any'))
_now = (fact.select('Purchasing_Document', 'Item', 'Schedule_Line', F.col('Order_Status_Purchase').alias('Fabric_Order_Status_Now'),
                    F.col('Execution_Status').alias('_ex_now'))
            .join(eket.select('Purchasing_Document', 'Item', 'Schedule_Line', 'Scheduled_Quantity', 'Quantity_Delivered'),
                  ['Purchasing_Document', 'Item', 'Schedule_Line'], 'left')
            .join(_item, ['Purchasing_Document', 'Item'], 'left')
            .join(ekpo.select('Purchasing_Document', 'Item', 'Delivery_Completed'), ['Purchasing_Document', 'Item'], 'left')
            .join(gr_item.select('Purchasing_Document', 'Item', 'Item_Gr_Qty', 'Last_GR_Date'), ['Purchasing_Document', 'Item'], 'left')
            .join(ekko.select('Purchasing_Document', F.col('Purch_Doc_Category').alias('Header_Category')), 'Purchasing_Document', 'left'))
_dA, _sq = F.col('Item_Delivered_A'), F.col('Item_Scheduled_Qty')
_exn = lambda c: (F.when(c.startswith('Cancelled'), 'Cancelled').when(F.lower(c) == 'partially open', 'Partially open').otherwise(c))
_now = (_now.withColumn('_ex_A', F.when(F.col('_ex_now').startswith('Cancelled'), F.col('_ex_now'))
                                  .when(F.col('Delivery_Completed') == 'X', 'Closed')
                                  .when((_dA > 0) & (_dA < _sq), 'Partially Open').when(_dA >= _sq, 'Closed').otherwise('Open'))
            .withColumn('Fabric_Execution_Now', _exn(F.col('_ex_now'))).withColumn('Fabric_Execution_A', _exn(F.col('_ex_A')))
            .drop('_ex_now', '_ex_A'))
_now = _k(_now, 'Purchasing_Document', 'Item', 'Schedule_Line')
cmp_ = _ea.join(_now.drop('Purchasing_Document', 'Item', 'Schedule_Line'), ['_po', '_it', '_sl'], 'left')
_f, _e, _o = F.col('Fabric_Order_Status_Now'), F.col('EA_Order_Status'), F.col('Fabric_Order_Status_16Nov')
_OPEN = ['To be confirmed', 'Confirmed', 'Ordered', 'Shipping notification', 'Goods issued', 'Outbound delivery created', 'Stock committed',
         'GR blocked stock (103)', 'GR blocked stock', 'Blocked', 'Service entry sheet created']
_CANC, _OC = ['Cancelled completely', 'Cancelled partially'], 'Closed by order creation or IR'
_sc, _rc, _gr = (F.coalesce(F.col(c).cast('double'), F.lit(0.0)) for c in ('Scheduled_Quantity', 'Quantity_Delivered', 'Item_Gr_Qty'))
_cause_now = (F.when(_f.isNull() | (_f == _e), F.lit(None))          # the same 13 causes as PROC_Order_Status_Differences.xlsx
    .when((F.col('Header_Category') == 'A') | (_f == 'To be ordered'), 10)
    .when((_sc == 0) & ((_e == '#N/A') | ((_f == _OC) & _e.isin(_OPEN))), 2)
    .when(_e == '#N/A', 13).when(_f == 'Return item', 11)
    .when((_e == 'Closed by GR') & (_f != 'Closed by GR') & (_rc == 0) & (_gr == 0), 1)
    .when((_e == 'Closed by GR') & _f.isin(_OPEN), 8)
    .when((_f == 'Closed by GR') & (_e == 'Outbound delivery created'), 4)
    .when(_f.isin(_CANC) | _e.isin(_CANC), 7)
    .when(((_f == 'Closed by GR') & (_e == _OC)) | ((_f == _OC) & (_e == 'Closed by GR')), 9)
    .when(_f.isin(_OPEN) & (_e == _OC), 6).when(_e == 'To be ordered', 12)
    .when(_f.isin(_OPEN) & _e.isin(_OPEN), 3).otherwise(13))
cmp_ = (cmp_.withColumn('Cause_Now', _cause_now)
    .withColumn('Result', F.when(_f.isNull(), 'No longer in Fabric').when((_o == _e) & (_f == _e), 'Same, unchanged')
                           .when(_f == _e, 'Now matches EA').when(_o == _e, 'Newly different').otherwise('Still different'))
    .withColumn('Receipt_From_16Nov', F.col('Last_GR_Date') >= F.lit('2026-11-16'))   # ▶ on or after the extract date (was after)
    .withColumn('Cause_16Nov', F.col('Cause_16Nov').cast('int'))).cache()
_n_new = _now.join(_ea.select('_po', '_it', '_sl'), ['_po', '_it', '_sl'], 'left_anti') \
             .join(ekko.filter((F.col('Created_On') >= F.lit(CREATED_FROM)) & (F.col('Created_On') <= F.lit(EXTRACT_TO)))
                       .select(F.regexp_replace('Purchasing_Document', '^0+', '').alias('_po')), '_po').count()
_tot = cmp_.agg(F.count('*').alias('n'), F.sum((_o == _e).cast('int')).alias('same_16'),
                F.sum(_f.isNotNull().cast('int')).alias('in_fabric'), F.sum((_f == _e).cast('int')).alias('same_now')).first()
print(f"Q12 · lines in the extract: {_tot['n']:,} | still in Fabric: {_tot['in_fabric']:,} | new lines on the same POs: {_n_new:,}")
print(f"Order status = EA: 16 Nov {_tot['same_16']:,} ({_tot['same_16'] / _tot['n']:.1%}) | now {_tot['same_now']:,} "
      f"({_tot['same_now'] / max(_tot['in_fabric'], 1):.1%} of the lines still in Fabric)")
_ok = F.col('EA_Execution_Line') != '#N/A'
_x = cmp_.filter(_ok & _f.isNotNull()).agg(F.count('*').alias('n'),
        F.sum((F.col('Fabric_Execution_16Nov') == F.col('EA_Execution_Line')).cast('int')).alias('m16'),
        F.sum((F.col('Fabric_Execution_Now') == F.col('EA_Execution_Line')).cast('int')).alias('mnow'),
        F.sum((F.col('Fabric_Execution_A') == F.col('EA_Execution_Line')).cast('int')).alias('mA')).first()
print(f"Execution status = EA (line), {_x['n']:,} lines: 16 Nov {_x['m16']:,} ({_x['m16'] / max(_x['n'], 1):.1%}) | now {_x['mnow']:,} "
      f"({_x['mnow'] / max(_x['n'], 1):.1%}) | with Option A {_x['mA']:,} ({_x['mA'] / max(_x['n'], 1):.1%})")
print('\n[Q12-1] Result per line'); cmp_.groupBy('Result').count().orderBy(F.desc('count')).show(truncate=False)
print('[Q12-2] Per 16 Nov cause')
(cmp_.filter(F.col('Cause_16Nov').isNotNull()).groupBy('Cause_16Nov', 'Cause_16Nov_Name')
     .agg(F.count('*').alias('lines_16Nov'), F.sum((F.col('Result') == 'Now matches EA').cast('int')).alias('now_match'),
          F.sum((F.col('Result') == 'Still different').cast('int')).alias('still_diff'),
          F.sum(((F.col('Result') == 'Still different') & (F.col('Cause_Now') != F.col('Cause_16Nov'))).cast('int')).alias('moved_cause'),
          F.sum((F.col('Result') == 'No longer in Fabric').cast('int')).alias('gone'),
          F.sum(F.coalesce(F.col('Receipt_From_16Nov'), F.lit(False)).cast('int')).alias('receipt_from_16Nov'))
     .orderBy(F.desc('lines_16Nov')).show(20, truncate=False))
print('[Q12-3] Still different and newly different, by cause now')
(cmp_.filter(F.col('Result').isin('Still different', 'Newly different')).groupBy('Cause_Now', 'Result').count()
     .orderBy('Cause_Now', 'Result').show(40, truncate=False))
print('[Q12-4] Fabric order status moves (16 Nov -> now)')
(cmp_.filter(_f.isNotNull() & (_o != _f)).groupBy('Fabric_Order_Status_16Nov', 'Fabric_Order_Status_Now', 'EA_Order_Status').count()
     .orderBy(F.desc('count')).show(40, truncate=False))
print('[Q12-5] Execution status: now -> with Option A, and EA (line)')
(cmp_.filter(_f.isNotNull() & (F.col('Fabric_Execution_Now') != F.col('Fabric_Execution_A')))
     .groupBy('Fabric_Execution_Now', 'Fabric_Execution_A', 'EA_Execution_Line').count().orderBy(F.desc('count')).show(30, truncate=False))
print(f'[Q12-6] Example lines: up to {N_LINE_EXAMPLES} per result and cause')
_w = Window.partitionBy('Result', F.coalesce('Cause_Now', 'Cause_16Nov')).orderBy('_po', '_it', '_sl')
(cmp_.filter(F.col('Result').isin('Now matches EA', 'Still different', 'Newly different', 'No longer in Fabric'))
     .withColumn('_rn', F.row_number().over(_w)).filter(F.col('_rn') <= N_LINE_EXAMPLES)
     .select('Result', 'Cause_16Nov', 'Cause_Now', 'PO', 'Item', 'Schedule_Line', 'EA_Order_Status', 'Fabric_Order_Status_16Nov',
             'Fabric_Order_Status_Now', F.col('Scheduled_Quantity').cast('decimal(18,3)').alias('Sched'),
             F.col('Quantity_Delivered').cast('decimal(18,3)').alias('Recv'), F.col('Item_Gr_Qty').cast('decimal(18,3)').alias('Item_GR'),
             'Last_GR_Date', 'EA_Execution_Line', 'Fabric_Execution_16Nov', 'Fabric_Execution_Now', 'Fabric_Execution_A')
     .orderBy('Result', F.coalesce('Cause_Now', 'Cause_16Nov'), 'PO', 'Item', 'Schedule_Line').show(2000, truncate=False))
print(f'[Q12-7] Example lines whose execution status changes with Option A (up to {N_LINE_EXAMPLES * 3})')
(cmp_.filter(_f.isNotNull() & (F.col('Fabric_Execution_Now') != F.col('Fabric_Execution_A')))
     .select('PO', 'Item', 'Schedule_Line', 'EA_Execution_Line', 'Fabric_Execution_Now', 'Fabric_Execution_A',
             F.col('Item_Delivered_All_Lines').cast('decimal(18,3)').alias('Item_Recv_All'), F.col('Item_Delivered_A').cast('decimal(18,3)').alias('Item_Recv_A'),
             F.col('Item_Scheduled_Qty').cast('decimal(18,3)').alias('Item_Sched'), F.col('Item_Gr_Qty').cast('decimal(18,3)').alias('Item_GR'))
     .orderBy('PO', 'Item', 'Schedule_Line').show(N_LINE_EXAMPLES * 3, truncate=False))
if SAVE_COMPARE_TABLE:
    (cmp_.drop('_po', '_it', '_sl').write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
         .save(tpath('gold', 'Chk_PROC_Order_Status_Compare')))
    register_table('gold', 'Chk_PROC_Order_Status_Compare')
    print('Line list written to Gold Chk_PROC_Order_Status_Compare (drop it when the review is done)')
# cmp_ stays cached for Q13, which releases it

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 30, Finished, Available, Finished, False)

Q12 · lines in the extract: 289,398 | still in Fabric: 260,003 | new lines on the same POs: 219
Order status = EA: 16 Nov 152,092 (52.6%) | now 225,177 (86.6% of the lines still in Fabric)
Execution status = EA (line), 259,748 lines: 16 Nov 154,031 (59.3%) | now 248,252 (95.6%) | with Option A 248,252 (95.6%)

[Q12-1] Result per line
+-------------------+------+
|Result             |count |
+-------------------+------+
|Same, unchanged    |140420|
|Now matches EA     |84757 |
|No longer in Fabric|29395 |
|Still different    |25915 |
|Newly different    |8911  |
+-------------------+------+

[Q12-2] Per 16 Nov cause
+-----------+-------------------------------+-----------+---------+----------+-----------+-----+------------------+
|Cause_16Nov|Cause_16Nov_Name               |lines_16Nov|now_match|still_diff|moved_cause|gone |receipt_from_16Nov|
+-----------+-------------------------------+-----------+---------+----------+-----------+-----+------------------+
|1          |Missing receipts

### Q13 · the evidence behind each remaining difference
Run after Q12 (it reads Q12's result). For the lines that still differ: whether SAP changed them from 16 Nov (then the extract's Every
Angle value is probably out of date), the execution-status pairs with the D2 line status, how much the stock transfers received,
delivery completed / under-delivery tolerance / invoice for causes 6 and 8, the EKES categories ZB / ZN for cause 3, deletion flags and
reversed receipts for causes 1 and 7, and what Every Angle shows for zero-quantity lines. Text only, nothing is written.

In [18]:
# ── Q13 · drill-down on the lines that still differ: the evidence behind each cause (19 Nov 2026) ─────────────────────────────
# Needs cmp_ from Q12 (same session). Reads only; every block prints one small table. "From 16 Nov" = on or after the extract date.
_FROM = '2026-11-16'
_nzk = lambda c: F.regexp_replace(F.trim(F.col(c).cast('string')), '^0+', '')
_bt = ekbe.withColumn('_q', F.col('Signed_Quantity').cast('double'))
_is_gr = F.col('Transevent_Type').eqNullSafe('1') | F.col('PO_History_Category').eqNullSafe('E')
_is_ir = F.col('Transevent_Type').eqNullSafe('2') | F.col('PO_History_Category').eqNullSafe('Q')
_hist = (_bt.groupBy('Purchasing_Document', 'Item')
            .agg(F.sum(_is_gr.cast('int')).alias('GR_Rows'), F.sum(F.when(_is_gr & (F.col('_q') < 0), 1).otherwise(0)).alias('GR_Reversal_Rows'),
                 F.sum(_is_ir.cast('int')).alias('IR_Rows'), F.sum(F.when(_is_ir, F.col('_q'))).alias('IR_Qty'),
                 F.max(F.when(_is_ir, F.col('Posting_Date'))).alias('Last_IR_Date')))
_conf = (ekes.groupBy('Purchasing_Document', 'Item')
             .agg(F.max(F.col('Confirmation_Category').isin('ZB').cast('int')).alias('Has_ZB'),
                  F.max(F.col('Confirmation_Category').isin('ZN').cast('int')).alias('Has_ZN'),
                  F.max('Created_On').alias('Last_Confirmation_Created')))
_ev = (ekpo.select('Purchasing_Document', 'Item', F.col('Deletion_Indicator').alias('Item_Deletion'), 'Final_Invoice',
                   'Underdelivery_Tolerance', 'Last_Changed_On')
           .join(ekko.select('Purchasing_Document', F.col('Deletion_Indicator').alias('Header_Deletion')), 'Purchasing_Document', 'left')
           .join(_hist, ['Purchasing_Document', 'Item'], 'left').join(_conf, ['Purchasing_Document', 'Item'], 'left')
           .withColumn('_po', _nzk('Purchasing_Document')).withColumn('_it', _nzk('Item')).drop('Purchasing_Document', 'Item'))
_d = (cmp_.join(_ev, ['_po', '_it'], 'left')
          .withColumn('_sc', F.coalesce(F.col('Scheduled_Quantity').cast('double'), F.lit(0.0)))
          .withColumn('_rc', F.coalesce(F.col('Quantity_Delivered').cast('double'), F.lit(0.0)))).cache()
_diff = _d.filter(F.col('Result').isin('Still different', 'Newly different'))
_since = (F.when(F.col('Last_GR_Date') >= F.lit(_FROM), '1 receipt from 16 Nov')
           .when(F.col('Last_Confirmation_Created') >= F.lit(_FROM), '2 confirmation / shipping notification from 16 Nov')
           .when(F.col('Last_Changed_On') >= F.lit(_FROM), '3 item changed from 16 Nov (e.g. cancelled)')
           .when(F.col('Last_IR_Date') >= F.lit(_FROM), '4 invoice from 16 Nov')
           .otherwise('5 no change found from 16 Nov'))
print('[Q13-1] Lines that differ now: did SAP change them from 16 Nov? (1-4 = the 16 Nov Every Angle value is probably out of date)')
(_diff.withColumn('Changed_From_16Nov', _since).groupBy('Result', 'Cause_Now', 'Changed_From_16Nov').count()
      .orderBy('Result', 'Cause_Now', 'Changed_From_16Nov').show(80, truncate=False))
print('[Q13-2] Execution status still different from EA (line), with the line-level status of decision D2')
_line_ex = (F.when(F.col('Fabric_Execution_Now') == 'Cancelled', 'Cancelled')
             .when((F.col('Delivery_Completed') == 'X') | ((F.col('_sc') - F.col('_rc')) <= OS_TOLERANCE), 'Closed')
             .when(F.col('_rc') > 0, 'Partially open').otherwise('Open'))
(_d.filter((F.col('EA_Execution_Line') != '#N/A') & F.col('Fabric_Execution_Now').isNotNull()
           & (F.col('Fabric_Execution_Now') != F.col('EA_Execution_Line')))
   .withColumn('D2_Line_Status', _line_ex)
   .groupBy('EA_Execution_Line', 'Fabric_Execution_Now', 'D2_Line_Status').count()
   .withColumn('D2_matches_EA', F.col('D2_Line_Status') == F.col('EA_Execution_Line'))
   .orderBy(F.desc('count')).show(30, truncate=False))
print('[Q13-3] Stock transfers that differ (cause 4 now): how much is received')
(_diff.filter(F.col('Cause_Now') == 4)
      .withColumn('Line_Received', F.when(F.col('_rc') >= F.col('_sc') - OS_TOLERANCE, 'line fully received')
                                    .when(F.col('_rc') > 0, 'line partly received').otherwise('nothing received on the line'))
      .withColumn('Item_Received', F.when(F.col('Item_Gr_Qty') >= F.col('Item_Scheduled_Qty') - OS_TOLERANCE, 'item fully received (EKBE)')
                                    .when(F.col('Item_Gr_Qty') > 0, 'item partly received').otherwise('no receipt on the item'))
      .groupBy('EA_Order_Status', 'Fabric_Order_Status_Now', 'Line_Received', 'Item_Received').count()
      .orderBy(F.desc('count')).show(20, truncate=False))
print('[Q13-4] Causes 6 and 8 (Every Angle closed, Fabric open): delivery completed, under-delivery tolerance, invoice')
_tol = F.coalesce(F.col('Underdelivery_Tolerance').cast('double'), F.lit(0.0))
(_diff.filter(F.col('Cause_Now').isin(6, 8))
      .withColumn('Delivery_Completed_X', F.coalesce(F.col('Delivery_Completed') == 'X', F.lit(False)))
      .withColumn('Within_Underdelivery_Tol', (_tol > 0) & (F.col('_rc') >= F.col('_sc') * (1 - _tol / 100) - OS_TOLERANCE) & (F.col('_rc') > 0))
      .withColumn('Has_Invoice', F.coalesce(F.col('IR_Rows'), F.lit(0)) > 0)
      .withColumn('Final_Invoice_X', F.coalesce(F.col('Final_Invoice') == 'X', F.lit(False)))
      .withColumn('Received_On_Line', F.col('_rc') > 0)
      .groupBy('Cause_Now', 'EA_Order_Status', 'Delivery_Completed_X', 'Within_Underdelivery_Tol', 'Has_Invoice', 'Final_Invoice_X',
               'Received_On_Line').count()
      .orderBy('Cause_Now', F.desc('count')).show(40, truncate=False))
print('[Q13-5] Open-line steps (cause 3 now): status pairs, and EKES categories that no status uses (question C1)')
(_diff.filter(F.col('Cause_Now') == 3)
      .withColumn('Has_ZB_or_ZN', (F.coalesce(F.col('Has_ZB'), F.lit(0)) + F.coalesce(F.col('Has_ZN'), F.lit(0))) > 0)
      .groupBy('EA_Order_Status', 'Fabric_Order_Status_Now', 'Has_ZB_or_ZN').count().orderBy(F.desc('count')).show(30, truncate=False))
print('[Q13-6] Cancelled (cause 7) and missing receipts (cause 1): deletion flags and reversed receipts')
(_diff.filter(F.col('Cause_Now').isin(1, 7))
      .withColumn('Item_Deleted_L', F.upper(F.trim(F.coalesce(F.col('Item_Deletion'), F.lit('')))) == 'L')
      .withColumn('Header_Deleted', F.upper(F.trim(F.coalesce(F.col('Header_Deletion'), F.lit('')))).isin('L', 'C'))
      .withColumn('Receipt_Reversed', (F.coalesce(F.col('GR_Reversal_Rows'), F.lit(0)) > 0) & (F.coalesce(F.col('Item_Gr_Qty'), F.lit(0.0)) <= 0))
      .withColumn('Received_On_Line', F.col('_rc') > 0)
      .groupBy('Cause_Now', 'EA_Order_Status', 'Fabric_Order_Status_Now', 'Item_Deleted_L', 'Header_Deleted', 'Receipt_Reversed',
               'Received_On_Line').count()
      .orderBy('Cause_Now', F.desc('count')).show(40, truncate=False))
print('[Q13-7] Zero-quantity lines of the extract: what Every Angle shows, and receipts found only on a zero line')
_z = _d.filter(F.col('Fabric_Order_Status_Now').isNotNull() & (F.col('_sc') == 0))
_z.groupBy('EA_Order_Status', 'Fabric_Order_Status_Now').count().orderBy(F.desc('count')).show(20, truncate=False)
_only = _z.filter((F.col('_rc') > 0) & ((F.coalesce(F.col('Item_Delivered_All_Lines'), F.lit(0.0)) - F.col('_rc')) <= OS_TOLERANCE)).count()
print(f'Zero-quantity lines: {_z.count():,} | with a receipt on no other line of the item: {_only:,} '
      '(removing zero lines from Fact_PO_LineItem_Schedule does not change item totals: NB_Gold_PROC cell 13 reads EKET)')
_d.unpersist(); cmp_.unpersist()

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 31, Finished, Available, Finished, False)

[Q13-1] Lines that differ now: did SAP change them from 16 Nov? (1-4 = the 16 Nov Every Angle value is probably out of date)
+---------------+---------+--------------------------------------------------+-----+
|Result         |Cause_Now|Changed_From_16Nov                                |count|
+---------------+---------+--------------------------------------------------+-----+
|Newly different|1        |1 receipt from 16 Nov                             |2    |
|Newly different|1        |5 no change found from 16 Nov                     |8    |
|Newly different|3        |1 receipt from 16 Nov                             |80   |
|Newly different|3        |2 confirmation / shipping notification from 16 Nov|1072 |
|Newly different|3        |3 item changed from 16 Nov (e.g. cancelled)       |134  |
|Newly different|3        |4 invoice from 16 Nov                             |14   |
|Newly different|3        |5 no change found from 16 Nov                     |857  |
|Newly different|4       

DataFrame[_po: string, _it: string, _sl: string, PO: string, Item: string, Schedule_Line: string, EA_Order_Status: string, EA_Execution_Line: string, EA_Execution_Header: string, Fabric_Order_Status_16Nov: string, Fabric_Execution_16Nov: string, Cause_16Nov: int, Cause_16Nov_Name: string, Rows_In_Extract: string, Fabric_Order_Status_Now: string, Scheduled_Quantity: decimal(15,3), Quantity_Delivered: decimal(15,3), Item_Delivered_All_Lines: decimal(25,3), Item_Scheduled_Qty: decimal(25,3), Item_Delivered_A: decimal(25,3), Delivery_Completed: string, Item_Gr_Qty: double, Last_GR_Date: date, Header_Category: string, Fabric_Execution_Now: string, Fabric_Execution_A: string, Cause_Now: int, Result: string, Receipt_From_16Nov: boolean]

### Q14 · the 18 Nov audit fields after the PROC Gold run
Fill rates of the new item columns, `Fact_PO_Header` totals against the sum of their items, `Invoice_Status` on both dimensions,
`First_GR_Date` against `Fact_OTD`, and the DQ results of the last PROC Gold run (including `zero_line_receipt` from Option A and
`base_unit_factor`). Text only, nothing is written.

In [19]:
# ── Q14 · the 18 Nov audit fields after the PROC Gold run: fill rates, totals, statuses, DQ results (19 Nov 2026) ─────────────────
# Reads Gold and OPS only. Expected: the header totals equal the sum of their items; no Closed / Cancelled item keeps an active open
# quantity; every item with a receipt in Fact_OTD has a First_GR_Date.
_ds, _hd = gold('Fact_PO_Delivery_Status'), gold('Fact_PO_Header')
_di, _dh = gold('Dim_PO_Item'), gold('Dim_PO_Header')
_want = ['Item_Value', 'Item_Value_LC', 'Price_Per_Unit_DC', 'Price_Per_Unit_LC', 'Material_Plant_Price_Per_Unit', 'Purchase_Value',
         'Material_Value', 'GR_Amount_LC', 'Received_Value_DC', 'Received_Quantity', 'Invoiced_Quantity', 'Ordered_Quantity_Base_UOM',
         'Received_Quantity_Base_UOM', 'Invoiced_Quantity_Base_UOM', 'Quantity_Open_Active', 'Base_Factor_PO', 'Base_Factor_MARM']
_have = [c for c in _want if c in _ds.columns]
_miss = [c for c in _want if c not in _ds.columns]
_n = _ds.count()
print(f'[Q14-1] Fact_PO_Delivery_Status: {_n:,} items | 18 Nov columns missing: {_miss or "none"}')
_fr = _ds.agg(*[F.sum(F.col(c).isNotNull().cast('int')).alias(c) for c in _have]).first()
for c in _have:
    print(f'  {c:<32} filled {_fr[c]:>12,}  ({_fr[c] / max(_n, 1):6.1%})')
_bf = F.coalesce(*[F.col(c) for c in ('Base_Factor_PO', 'Base_Factor_MARM') if c in _ds.columns]) \
      if any(c in _ds.columns for c in ('Base_Factor_PO', 'Base_Factor_MARM')) else F.lit(None).cast('double')
_closed = F.col('Execution_Status').isin('Closed', 'Cancelled_Header', 'Cancelled_Line')
_c = _ds.agg(F.sum((F.col('Order_Quantity').isNotNull() & _bf.isNull()).cast('int')).alias('no_factor'),
             F.sum((_closed & (F.coalesce(F.col('Quantity_Open_Active'), F.lit(0.0)) != 0)).cast('int')).alias('qoa_bad'),
             F.sum((_closed & (F.coalesce(F.col('Quantity_Open'), F.lit(0.0)) != 0)).cast('int')).alias('qo_closed')).first()
print(f"  Items without a base-unit factor: {_c['no_factor']:,} | Closed / Cancelled items with Quantity_Open_Active <> 0: "
      f"{_c['qoa_bad']:,} (expected 0) | Closed / Cancelled items whose Quantity_Open <> 0 (the old mismatch, now 0 in the active column): {_c['qo_closed']:,}")
print('[Q14-2] Fact_PO_Header totals against the sum of their items (POs that differ by more than 0.01)')
_del = F.upper(F.trim(F.coalesce(F.col('Deletion_Indicator'), F.lit('')))) == 'L'
_sum = {'Net_Order_Value': F.sum('Net_Order_Value'), 'Purchase_Value': F.sum('Purchase_Value'), 'Material_Value': F.sum('Material_Value'),
        'Ordered_Quantity_Base_UOM': F.sum(F.when(~_del, F.col('Ordered_Quantity_Base_UOM'))),
        'Received_Quantity_Base_UOM': F.sum('Received_Quantity_Base_UOM'), 'Invoiced_Quantity_Base_UOM': F.sum('Invoiced_Quantity_Base_UOM'),
        'Quantity_Open_Active_Base_UOM': F.sum(F.col('Quantity_Open_Active') * _bf)}
_sum = {k: v for k, v in _sum.items() if k in _hd.columns}
_it = _ds.groupBy('PO_SK').agg(*[v.alias(f'_i_{k}') for k, v in _sum.items()])
_j = _hd.select('PO_SK', *_sum.keys()).join(_it, 'PO_SK', 'left')
_r = _j.agg(*[F.sum((F.abs(F.coalesce(F.col(k), F.lit(0.0)) - F.coalesce(F.col(f'_i_{k}'), F.lit(0.0))) > 0.01).cast('int')).alias(k)
              for k in _sum]).first()
for k in _sum:
    print(f'  {k:<32} POs that differ: {_r[k]:,}')
print('[Q14-3] Invoice_Status on Dim_PO_Item and Dim_PO_Header (header NULL split by why: no item / every item deleted)')
if 'Invoice_Status' in _di.columns: _di.groupBy('Invoice_Status').count().orderBy(F.desc('count')).show(truncate=False)
if 'Invoice_Status' in _dh.columns: _dh.groupBy('Invoice_Status').count().orderBy(F.desc('count')).show(truncate=False)
if 'Invoice_Status' in _dh.columns:
    _itm_n = _ds.groupBy('PO_SK').agg(F.count('*').alias('_items'),
                                      F.sum((F.upper(F.trim(F.coalesce(F.col('Deletion_Indicator'), F.lit('')))) == 'L').cast('int')).alias('_deleted'))
    (_dh.filter(F.col('Invoice_Status').isNull()).join(_itm_n, 'PO_SK', 'left')
        .withColumn('Why', F.when(F.col('_items').isNull(), 'PO has no item').when(F.col('_deleted') == F.col('_items'), 'every item deleted (L)')
                             .otherwise('other'))
        .groupBy('Why').count().show(truncate=False))
print('[Q14-4] First_GR_Date: items with a receipt in Fact_OTD but no date, and header date against the earliest item date')
_otd_items = gold('Fact_OTD').select('PO_Item_SK').distinct()
_no_date = _di.join(_otd_items, 'PO_Item_SK').filter(F.col('First_GR_Date').isNull()).count() if 'First_GR_Date' in _di.columns else None
_di_po = _di.select('PO_Item_SK', 'First_GR_Date').join(_ds.select('PO_Item_SK', 'PO_SK'), 'PO_Item_SK')   # Dim_PO_Item has no PO_SK
_hdr_cmp = (_dh.select('PO_SK', F.col('First_GR_Date').alias('_h'))
               .join(_di_po.groupBy('PO_SK').agg(F.min('First_GR_Date').alias('_i')), 'PO_SK', 'left')
               .filter(~F.col('_h').eqNullSafe(F.col('_i'))).count()) if 'First_GR_Date' in _dh.columns else None
print(f'  Items with a Fact_OTD receipt but no First_GR_Date: {_no_date} (expected 0) | POs whose date differs from their earliest item: {_hdr_cmp} (expected 0)')
print('[Q14-5] DQ results of the last PROC Gold run (from OPS dq_results)')
try:
    _dq = read_table('ops', 'dq_results')
    _last = (_dq.filter((F.upper(F.col('layer')) == 'GOLD') & F.col('table_name').isin('Fact_PO_Delivery_Status', 'Fact_PO_LineItem_Schedule'))
                .orderBy(F.desc('logged_at')).select('pipeline_run_id', 'logged_at').first())
    if _last:
        print(f"  Run {_last['pipeline_run_id']} ({_last['logged_at']})")
        (_dq.filter(F.col('pipeline_run_id') == _last['pipeline_run_id'])
            .select('table_name', 'check_name', 'status', 'observed', F.substring('message', 1, 140).alias('message'))
            .orderBy(F.desc('status'), 'table_name', 'check_name').show(80, truncate=False))
    else:
        print('  No Gold DQ rows for the PROC facts found')
except Exception as e:
    print(f'  dq_results not readable here ({str(e)[:150]})')

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 32, Finished, Available, Finished, False)

[Q14-1] Fact_PO_Delivery_Status: 6,039,284 items | 18 Nov columns missing: ['Base_Factor_PO', 'Base_Factor_MARM']
  Item_Value                       filled    6,039,284  (100.0%)
  Item_Value_LC                    filled    6,024,985  ( 99.8%)
  Price_Per_Unit_DC                filled            0  (  0.0%)
  Price_Per_Unit_LC                filled            0  (  0.0%)
  Material_Plant_Price_Per_Unit    filled    4,819,941  ( 79.8%)
  Purchase_Value                   filled    5,690,729  ( 94.2%)
  Material_Value                   filled            0  (  0.0%)
  GR_Amount_LC                     filled    5,557,962  ( 92.0%)
  Received_Value_DC                filled    2,758,230  ( 45.7%)
  Received_Quantity                filled    5,557,962  ( 92.0%)
  Invoiced_Quantity                filled    3,632,506  ( 60.1%)
  Ordered_Quantity_Base_UOM        filled            0  (  0.0%)
  Received_Quantity_Base_UOM       filled            0  (  0.0%)
  Invoiced_Quantity_Base_UOM       filled

### Q15 · causes 6 and 8: which rule explains Every Angle's "closed"
Run after Q12 and Q13. For cause 8 (Every Angle Closed by GR on partly received lines) and cause 6 (Every Angle Closed by order creation
or IR with no goods receipt), each candidate rule is scored twice: how many cause lines it explains, and how many control lines (Every
Angle and Fabric show the same open step) it would close by mistake. A rule worth proposing to Jeremy and Procurement team explains
most cause lines and closes almost no control lines. Text only, nothing is written.

In [20]:
# ── Q15 · causes 6 and 8: which rule would explain Every Angle's "closed", tested against lines Every Angle keeps open (20 Nov 2026) ─
# Needs Q12 (cmp_) and Q13 (_nzk) in the same session. Reads only. For each candidate rule: how many cause lines it explains, and how
# many control lines (Every Angle and Fabric both show an open step) it would close by mistake. A good rule: high first, near 0 second.
_tolq = OS_TOLERANCE
_ek = (eket.select(_nzk('Purchasing_Document').alias('_po'), _nzk('Item').alias('_it'), _nzk('Schedule_Line').alias('_sl'), 'Delivery_Date',
                   F.coalesce(F.col('Scheduled_Quantity').cast('double'), F.lit(0.0)).alias('_sc'),
                   F.coalesce(F.col('Quantity_Delivered').cast('double'), F.lit(0.0)).alias('_rc'),
                   F.coalesce(F.col('Qty_Reduced_MRP').cast('double'), F.lit(0.0)).alias('_red'))
          .withColumn('_cum_sched', F.sum('_sc').over(Window.partitionBy('_po', '_it').orderBy('Delivery_Date', F.col('_sl').cast('int'))
                                                     .rowsBetween(Window.unboundedPreceding, Window.currentRow))))
_iv = (ekbe.filter(F.col('Transevent_Type').eqNullSafe('2') | F.col('PO_History_Category').eqNullSafe('Q'))
           .groupBy(_nzk('Purchasing_Document').alias('_po'), _nzk('Item').alias('_it'))
           .agg(F.sum(F.col('Signed_Quantity').cast('double')).alias('_iir')))
_gq = gr_item.select(_nzk('Purchasing_Document').alias('_po'), _nzk('Item').alias('_it'), F.col('Item_Gr_Qty').cast('double').alias('_igr'))
_ip = ekpo.select(_nzk('Purchasing_Document').alias('_po'), _nzk('Item').alias('_it'), F.col('Order_Quantity').cast('double').alias('_ord'),
                  F.upper(F.trim(F.coalesce(F.col('Goods_Receipt_Ind'), F.lit('')))).alias('_wepos'),
                  F.upper(F.trim(F.coalesce(F.col('Item_Category'), F.lit('')))).alias('_pstyp'),
                  F.upper(F.trim(F.coalesce(F.col('Acct_Assignment_Cat'), F.lit('')))).alias('_knttp'))
_x = (cmp_.select('_po', '_it', '_sl', 'Result', 'Cause_Now', 'EA_Order_Status', 'Fabric_Order_Status_Now', 'PO', 'Item', 'Schedule_Line')
          .filter(F.col('Fabric_Order_Status_Now').isNotNull())
          .join(_ek, ['_po', '_it', '_sl'], 'left').join(_iv, ['_po', '_it'], 'left').join(_gq, ['_po', '_it'], 'left')
          .join(_ip, ['_po', '_it'], 'left')
          .fillna(0.0, ['_iir', '_igr'])).cache()
_both_open = F.col('EA_Order_Status').isin(_OPEN) & F.col('Fabric_Order_Status_Now').isin(_OPEN) & (F.col('EA_Order_Status') == F.col('Fabric_Order_Status_Now'))
_diff_now = F.col('Result').isin('Still different', 'Newly different')
H8 = {'H1 FIFO: item receipts cover the lines up to this one': F.col('_igr') >= F.col('_cum_sched') - _tolq,
      'H2 item fully received (receipts >= ordered)': F.col('_igr') >= F.col('_ord') - _tolq,
      'H3 line received + reduced by MRP >= line quantity': (F.col('_red') > 0) & (F.col('_rc') + F.col('_red') >= F.col('_sc') - _tolq),
      'H4 item invoices cover the lines up to this one': F.col('_iir') >= F.col('_cum_sched') - _tolq}
H6 = {'H1 no goods receipt expected (EKPO WEPOS blank)': F.col('_wepos') == '',
      'H2 item fully invoiced (invoices >= ordered)': F.col('_iir') >= F.col('_ord') - _tolq,
      'H3 any invoice on the item': F.col('_iir') > 0,
      'H4 service item (category D)': F.col('_pstyp') == 'D',
      'H5 account-assigned (EKPO KNTTP not blank)': F.col('_knttp') != '',
      'H6 no goods receipt expected AND item fully invoiced': (F.col('_wepos') == '') & (F.col('_iir') >= F.col('_ord') - _tolq)}
def _score(name, target, control, hyps):
    t, c = _x.filter(target), _x.filter(control)
    nt, nc = t.count(), c.count()
    rt = t.agg(*[F.sum(e.cast('int')).alias(k) for k, e in hyps.items()]).first()
    rc = c.agg(*[F.sum(e.cast('int')).alias(k) for k, e in hyps.items()]).first()
    print(f'\n[{name}] target lines: {nt:,} | control lines: {nc:,}')
    print(f"  {'Candidate rule':<62} {'explains':>14} {'closes control by mistake':>28}")
    for k in hyps:
        a, b = rt[k] or 0, rc[k] or 0
        print(f"  {k:<62} {a:>8,} {a / max(nt, 1):>5.0%} {b:>21,} {b / max(nc, 1):>5.0%}")
_score('Q15-1 Cause 8: Every Angle Closed by GR, Fabric open on a partly received line',
       _diff_now & (F.col('Cause_Now') == 8),
       _both_open & (F.col('_rc') > 0) & (F.col('_rc') < F.col('_sc') - _tolq), H8)
print('[Q15-2] Cause 8 examples')
(_x.filter(_diff_now & (F.col('Cause_Now') == 8))
   .select('PO', 'Item', 'Schedule_Line', 'EA_Order_Status', 'Fabric_Order_Status_Now', 'Delivery_Date', '_sc', '_rc', '_red', '_cum_sched', '_igr',
           '_ord', '_iir').orderBy('PO', 'Item', 'Schedule_Line').show(15, truncate=False))
_score('Q15-3 Cause 6: Every Angle Closed by order creation or IR, Fabric open, no receipt on the line',
       _diff_now & (F.col('Cause_Now') == 6),
       _both_open & (F.col('_rc') == 0) & (F.col('_iir') > 0), H6)
print('[Q15-4] Cause 6 examples')
(_x.filter(_diff_now & (F.col('Cause_Now') == 6))
   .select('PO', 'Item', 'Schedule_Line', 'EA_Order_Status', 'Fabric_Order_Status_Now', '_sc', '_rc', '_igr', '_ord', '_iir', '_wepos', '_pstyp',
           '_knttp').orderBy('PO', 'Item', 'Schedule_Line').show(15, truncate=False))
print('[Q15-5] Cause 6: goods-receipt indicator, item category and account assignment')
(_x.filter(_diff_now & (F.col('Cause_Now') == 6)).groupBy('_wepos', '_pstyp', '_knttp').count().orderBy(F.desc('count')).show(15, truncate=False))
_x.unpersist()

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 33, Finished, Available, Finished, False)


[Q15-1 Cause 8: Every Angle Closed by GR, Fabric open on a partly received line] target lines: 1,251 | control lines: 148
  Candidate rule                                                       explains    closes control by mistake
  H1 FIFO: item receipts cover the lines up to this one                 0    0%                     0    0%
  H2 item fully received (receipts >= ordered)                          0    0%                     0    0%
  H3 line received + reduced by MRP >= line quantity                  267   21%                    39   26%
  H4 item invoices cover the lines up to this one                     111    9%                    10    7%
[Q15-2] Cause 8 examples
+--------+----+-------------+---------------+-----------------------+-------------+-------+-------+-------+----------+-------+-------+-------+
|PO      |Item|Schedule_Line|EA_Order_Status|Fabric_Order_Status_Now|Delivery_Date|_sc    |_rc    |_red   |_cum_sched|_igr   |_ord   |_iir   |
+--------+----+----------

DataFrame[_po: string, _it: string, _sl: string, Result: string, Cause_Now: int, EA_Order_Status: string, Fabric_Order_Status_Now: string, PO: string, Item: string, Schedule_Line: string, Delivery_Date: date, _sc: double, _rc: double, _red: double, _cum_sched: double, _iir: double, _igr: double, _ord: double, _wepos: string, _pstyp: string, _knttp: string]

In [21]:
# ── End-of-run report: result, issues in plain language, tables, checks ────────
end_of_run_report()

StatementMeta(, 9754aac8-1c2b-4b12-bcf2-7dc5d934da67, 34, Finished, Available, Finished, False)

════════════════════════════════════════════════════════════════════════════════════════════════════
 PROC_Check_Feedback_V4  |  run 618b50e1-caec-4134-850e-198c55001fa4
 20 Nov 2026 00:13–00:22 UTC (8m 38s)  |  0 of 0 tables OK  |  0 checks
 RESULT: ✓ All 0 tables loaded, all checks passed
════════════════════════════════════════════════════════════════════════════════════════════════════
 Details: SELECT * FROM dq_results WHERE pipeline_run_id = '618b50e1-caec-4134-850e-198c55001fa4' ORDER BY status
════════════════════════════════════════════════════════════════════════════════════════════════════
Session stopped to release capacity (END_SESSION = False keeps it)
  Session not stopped (__$$_STOP_SESSION_REQUEST_$$__)


'SUCCEEDED'